In [43]:
%pip install esda libpysal
%pip install folium

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [44]:
import pandas as pd
import numpy as np

import html
import folium

import warnings
from scipy.spatial import cKDTree
 
from libpysal.weights import DistanceBand
from esda import Moran, Join_Counts

In [45]:
df = pd.read_csv("saihat_google_maps_deduplicated.csv")
print(len(df["category"].unique()))

291


Since we have 291 categories of locations, any visualization  with this many unique categories will be unreadable.
To fix this, we will map the current categories to 20 more general parent categories.

In [46]:
# --- parent labels -----------------------------------------------------------
FOOD          = 'مطاعم ومقاهي'
GROCERY       = 'بقالة وأسواق غذائية'
RETAIL        = 'تجزئة وتسوق'
HEALTH        = 'صحة وطب'
BEAUTY        = 'تجميل وعناية شخصية'
AUTO          = 'سيارات ومركبات'
EDU           = 'تعليم'
SPORT         = 'رياضة وترفيه'
RELIGION      = 'أماكن دينية'
REALESTATE    = 'عقارات'
PROF          = 'خدمات مهنية وشخصية'
CONSTRUCTION  = 'إنشاءات وتحسين المنزل'
INDUSTRY      = 'صناعة وجملة ولوجستيات'
TRAVEL        = 'سفر وإقامة'
FINANCE       = 'خدمات مالية'
PETS          = 'حيوانات أليفة وبيطرة'
GOV           = 'جهات حكومية ومجتمعية'
AGRI          = 'زراعة'
OTHER         = 'أخرى'
 
# --- explicit category -> parent lookup -------------------------------------
PARENT_BY_CATEGORY = {
    # Food & Drink
    'مخبز لبيع الجملة': FOOD, 'مخبز': FOOD, 'متجر حلويات': FOOD,
    'مطعم للإفطار': FOOD, 'مطعم': FOOD, 'المعجنات': FOOD, 'متجر الكعك': FOOD,
    'كافيه': FOOD, 'متجر عصائر': FOOD, 'مطعم همبرغر': FOOD,
    'مطعم مأكولات إيطالية': FOOD, 'مطعم مأكولات مشوية': FOOD,
    'مطعم مخبوزات محلاة': FOOD, 'مطعم مأكولات صحية': FOOD, 'مقهى': FOOD,
    'متجر شيكولاتة': FOOD, 'مطعم وجبات خفيفة': FOOD, 'مطعم أسماك': FOOD,
    'مقهى لشرب الشاي': FOOD, 'بوفيه الحلويات وأطباق الحلو': FOOD,
    'مطعم الحلويات اليابانية': FOOD, 'الحلويات': FOOD, 'بوظة': FOOD,
    'مقهى تقليدي': FOOD, 'بار مشروبات': FOOD, 'مطعم عائلي': FOOD,
    'مطعم مأكولات بحرية': FOOD, 'مطعم مأكولات تركية': FOOD,
    'مطعم مأكولات الشرق الأوسط': FOOD, 'مطعم دجاج': FOOD, 'متجر القهوة': FOOD,
    'مطعم وجبات سريعة': FOOD,
 
    # Groceries & Food Markets
    'ملحمة': GROCERY, 'سوبرماركت': GROCERY, 'متجر بقالة': GROCERY,
    'سوق ضخمة': GROCERY, 'سوبر ماركت عروض الخصم': GROCERY,
    'متجر لبيع المكسرات': GROCERY, 'متجر فواكه وخضروات': GROCERY,
    'بائع خضار': GROCERY, 'متجر أسماك': GROCERY, 'سوق مأكولات بحرية': GROCERY,
    'تجهيز الأسماك': GROCERY,
 
    # Retail & Shopping
    'متجر سلع منزلية': RETAIL, 'متجر أجهزة إلكترونية': RETAIL,
    'متجر مفروشات': RETAIL, 'مركز تسوق': RETAIL, 'متجر': RETAIL,
    'متجر مستلزمات الخَبز': RETAIL, 'متجر دخان': RETAIL,
    'متجر مستحضرات الصحة والتجميل': RETAIL, 'متجر الأطفال': RETAIL,
    'متجر أدوات مكتبية': RETAIL, 'متجر مستلزمات المدارس': RETAIL,
    'متجر كتب': RETAIL, 'منفذ بيع بالتجزئة': RETAIL,
    'متجر مستلزمات فنية': RETAIL, 'متجر ملابس رجالي': RETAIL,
    'متجر ملابس': RETAIL, 'متجر ملابس أطفال': RETAIL, 'متجر ملابس رسمية': RETAIL,
    'متجر الملابس الداخلية': RETAIL, 'متجر أقمشة': RETAIL,
    'متجر أكسسوارات': RETAIL, 'متجر للزي الموحد': RETAIL, 'محل هدايا': RETAIL,
    'متجر الملابس الخارجية': RETAIL, 'متجر زهور': RETAIL, 'سوق ملابس': RETAIL,
    'متجر هواتف جوالة': RETAIL, 'متجر عطور': RETAIL, 'متجر ملابس الشباب': RETAIL,
    'متجر أدوات مطبخ': RETAIL, 'آلة بيع منتجات التجميل': RETAIL,
    'متجر أثاث': RETAIL, 'سوق': RETAIL, 'متجر للمبخرات': RETAIL,
    'متجر دراجات': RETAIL, 'متجر مجوهرات': RETAIL, 'متجر الملابس': RETAIL,
    'متجر مستحضرات تجميل': RETAIL, 'متجر لملحقات الهواتف الخلوية': RETAIL,
    'متجر مستلزمات المطابخ': RETAIL, 'متجر ألعاب': RETAIL, 'توصيل زهور': RETAIL,
    'متجر الأواني الزجاجية': RETAIL, 'متجر نظارات شمسية': RETAIL,
    'متجر ألعاب فيديو': RETAIL, 'متجر مستلزمات الأطفال': RETAIL,
    'مستلزمات الصيد البحري': RETAIL, 'سوق الزهور': RETAIL, 'مصمم زهور': RETAIL,
    'متجر ملابس رياضية': RETAIL,
 
    # Health & Medical
    'المكتب الطبي': HEALTH, 'مركز طبي': HEALTH, 'صيدلية': HEALTH,
    'مستشفى': HEALTH, 'المركز الطبي العام': HEALTH, 'مستشفى عام': HEALTH,
    'عِيادة أسنان': HEALTH, 'مركز الرعاية الصحية للأطفال': HEALTH,
    'مركز إعادة التأهيل': HEALTH, 'مركز الحفاظ على الصحة': HEALTH,
    'عيادة علاج طبيعي': HEALTH, 'دار لرعاية المرضى والمسنّين': HEALTH,
    'مركز الغسيل الكلوي': HEALTH, 'مختبر طبي': HEALTH, 'طبيب أسنان': HEALTH,
    'متجر للفيتامينات والمكملات الغذائية': HEALTH, 'متجر الأعشاب الطبية': HEALTH,
    'محل بصريات': HEALTH,
 
    # Beauty & Personal Care
    'صالون حلاقة': BEAUTY, 'صالون تجميل': BEAUTY, 'مصفف الشعر': BEAUTY,
    'فنان مكياج': BEAUTY, 'حديقة المنتجع الصحي': BEAUTY,
    'صالون عناية بالأظافر': BEAUTY, 'أخصائي مساج رياضي': BEAUTY,
 
    # Automotive
    'تصليح سيارات': AUTO, 'تاجر سيارات': AUTO, 'سوق سيارات': AUTO,
    'مركز صيانة سيارات': AUTO, 'متجر كماليات السيارات': AUTO,
    'غسيل السيارات بالخدمة الذاتية': AUTO, 'محطة غسل سيارات': AUTO,
    'ورشة إصلاح سيارات': AUTO, 'خدمة ضبط السيارات': AUTO,
    'ورشة ميكانيكا سيارات': AUTO, 'متجر إطارات': AUTO, 'منجد سيارات': AUTO,
    'ورشة إصلاح هياكل سيارات': AUTO, 'تاجر جملة اكسسوارات سيارات': AUTO,
    'خدمة تغيير الزيت': AUTO, 'تصليح دراجات نارية': AUTO,
    'خدمة سحب سيارات': AUTO, 'متجر قطع غيار سيارات': AUTO,
    'متجر لإصلاح الدراجات الرباعية': AUTO,
    'خدمة العناية الشاملة بالسيارة': AUTO, 'تاجر سيارات مستعملة': AUTO,
    'مورد سيارات هوندا': AUTO, 'تاجر السيارات': AUTO, 'مورد سيارات مازدا': AUTO,
    'محطة وقود': AUTO, 'موقف سيارات': AUTO,
 
    # Education
    'مؤسسة تعليمية': EDU, 'مركز تدريب': EDU,
    'مدرسة تعليم اللغة الإنجليزية': EDU, 'مركز تدريب مهني': EDU,
    'مدرسة ثانوية': EDU, 'المركز التعليمي': EDU, 'مكتبة عامة': EDU,
    'مركز لرعاية الأطفال': EDU, 'مدرسة خاصة': EDU, 'مدرسة تعليم عام': EDU,
    'مدرسة تعليم خاص': EDU, 'مدرسة إعدادية': EDU, 'مدرسة ابتدائية': EDU,
    'المدرسة الثانوية للبنين': EDU, 'المرحلة الابتدائية': EDU,
    'حضانة أطفال': EDU, 'المدرسة الثانوية للبنات': EDU,
    'مدرسة ابتدائية إعدادية': EDU, 'رياض أطفال': EDU, 'حضانة': EDU,
 
    # Sports & Recreation
    'صالة رياضة': SPORT, 'النادي الرياضي': SPORT, 'نادي رياضة البادل تنس': SPORT,
    'صالة رياضة البيلاتس': SPORT, 'غرفة لياقة': SPORT, 'قاعة زفاف': SPORT,
    'صالة يوغا': SPORT, 'قاعة مناسبات': SPORT, 'قاعة حفلات': SPORT,
    'مجمع رياضي': SPORT, 'ملعب كرة قدم': SPORT, 'نادي البلياردو': SPORT,
    'ملعب كرة سلة': SPORT, 'مركز ترفيهي': SPORT, 'مدينة ملاهي': SPORT,
    'نادي رياضي': SPORT, 'حديقة': SPORT, 'مركز ترفيهي للأطفال': SPORT,
    'نادي أطفال': SPORT, 'الملعب المغُطى': SPORT, 'متنزه': SPORT,
    'متنزه عام': SPORT, 'متنزه المدينة': SPORT, 'حديقة مجتمعية': SPORT,
    'ملعب': SPORT, 'ملعب رياضي': SPORT, 'مركز ترفيه': SPORT,
    'نادي كرة القدم': SPORT, 'نادي كرة قدم': SPORT, 'نادي الفنون القتالية': SPORT,
 
    # Religious
    'مسجد': RELIGION, 'مكان عبادة': RELIGION, 'الوجهة الدينية': RELIGION,
    'مؤسسة دينية': RELIGION,
 
    # Real Estate
    'مجمع سكني': REALESTATE, 'مطوّر عقارات': REALESTATE, 'مبنى سكني': REALESTATE,
    'وكالة عقارات تجارية': REALESTATE, 'وكالة عقارات': REALESTATE,
    'وكيل عقاري': REALESTATE, 'شركة إدارة عقارات': REALESTATE,
    'مستشار عقارات': REALESTATE, 'مثمّن عقارات': REALESTATE,
    'التنمية السكنية': REALESTATE,
 
    # Professional & Personal Services
    'جهة توظيف': PROF, 'مكتب الشركات': PROF, 'خدمات قانونية': PROF,
    'خدمة إصلاح المعدات المكتبية': PROF, 'خدمة تنظيف منازل': PROF,
    'مستشار': PROF, 'محامٍ': PROF, 'مكتب محاماة قانون مدني': PROF,
    'مستشار الضرائب': PROF, 'خدمة إصلاح الطابعات': PROF, 'خياط سيدات': PROF,
    'تصوير': PROF, 'خيّاط ملابس الرجال': PROF, 'مغسلة': PROF, 'مصمم داخلي': PROF,
    'خياط': PROF, 'محل خياطة': PROF, 'خيّاط الملابس المفصّلة': PROF,
    'منظم حفلات الزفاف': PROF, 'خدمات تنظيف الملابس': PROF, 'وكالة إعلانية': PROF,
    'منظم أحداث': PROF, 'حرفي': PROF,
 
    # Construction & Home Improvement
    'المقاول': CONSTRUCTION, 'حداد': CONSTRUCTION,
    'تاجر جملة مواد البناء': CONSTRUCTION, 'مورّد المظلات': CONSTRUCTION,
    'مقاول رخام': CONSTRUCTION, 'متجر مستلزمات السباكة': CONSTRUCTION,
    'بناء منازل': CONSTRUCTION, 'مقاول تكييف هواء': CONSTRUCTION,
    'متجر بلاط': CONSTRUCTION, 'مقاول عام': CONSTRUCTION, 'متجر خردوات': CONSTRUCTION,
    'متجر أوراق حائط': CONSTRUCTION, 'صانع ومورد ستائر': CONSTRUCTION,
    'متجر أخشاب': CONSTRUCTION,
 
    # Industrial, Wholesale & Logistics
    'مورّد زجاجات المياه': INDUSTRY, 'مستودع': INDUSTRY, 'مصنع': INDUSTRY,
    'مورّد المنتجات الغذائية': INDUSTRY, 'شركة الحافلات': INDUSTRY,
    'وكالة تأجير معدات': INDUSTRY, 'مورّد الثلج': INDUSTRY,
    'مورد غاز التروبين (البروبان)': INDUSTRY, 'شركة مرافق الكهرباء': INDUSTRY,
    'خدمات الشحن والتخليص الجمركي': INDUSTRY, 'خدمة شحن': INDUSTRY,
    'شحن وخدمات بريدية': INDUSTRY, 'خدمة نقل': INDUSTRY,
    'خدمة وسائل النقل': INDUSTRY, 'شركة طاقة شمسية': INDUSTRY,
    'تاجر جملة': INDUSTRY, 'مصنع الشيكولاتة': INDUSTRY, 'صانع أثاث': INDUSTRY,
    'مصنع أثاث': INDUSTRY, 'شركة تصنيع البلاستيك': INDUSTRY,
    'مصنّع منتجات بصرية': INDUSTRY,
 
    # Travel & Lodging
    'مكتب سفريات': TRAVEL, 'وكالة سياحية': TRAVEL, 'فندق': TRAVEL,
    'تأجير كوخ': TRAVEL, 'دار ضيافة': TRAVEL,
 
    # Finance
    'مصرف': FINANCE, 'ماكينة صراف آلي': FINANCE,
 
    # Pets & Veterinary
    'خدمة طوارئ بيطرية': PETS, 'متجر طيور': PETS,
    'متجر مستلزمات حيوانات أليفة': PETS, 'مستشفى بيطري': PETS,
    'الصيدلة البيطرية': PETS,
 
    # Government & Community
    'جمعية / منظمة': GOV, 'مكتب بريد': GOV, 'الدفاع المدني': GOV,
    'مركز إطفاء': GOV, 'منظمة خدمات اجتماعية': GOV, 'مؤسسة غير ربحية': GOV,
    'مجلس': GOV,
 
    # Agriculture
    'مزرعة': AGRI, 'مزرعة حيوانات': AGRI, 'كرم عنب': AGRI,
    'إسطبل تربية الخيول': AGRI,
 
    # Too generic to classify confidently
    'مؤسسة': OTHER,
}


def map_to_parent(category):
    """Return the parent group for a single raw category value."""
    if not isinstance(category, str) or not category.strip():
        return pd.NA
 
    cat = category.strip()
    if cat in PARENT_BY_CATEGORY:
        return PARENT_BY_CATEGORY[cat]
 
    return OTHER

In [47]:
df_with_parent = df.copy()
df_with_parent['category_parent'] = df['category'].map(map_to_parent)
df_with_parent.head()

,Unnamed: 0,name_ar,category,full_address,phone_number,website,rating,plus_code,opening_times,opening_status,price_range,city_ar,region_ar,latitude,longitude,google_maps_url,google_maps_url_raw,completion_score,category_parent
0,0,مجالات التميز للاستقدام,جهة توظيف,"Dist, 2884 طريق الملك عبدالعزيز، المنتزه، سيها...",058 364 6339,NaN,4.2,F2RP+VW,"{""الأحد"": ""١٠:٠٠ص–١١:٣٠م""}",مفتوح · يغلق عند الساعة ١١:٣٠ م,NaN,سيهات,المنطقة الشرقية,26.492194,50.037329,https://www.google.com/maps?ftid=0x3e49ffe29a8...,https://www.google.com/maps/place/%D9%85%D8%AC...,14,خدمات مهنية وشخصية
1,1,مكتب بحار العالم للاسقدام,جهة توظيف,عمر بن عبد العزيز، حي النور، سيهات‎ 32452,050 561 0083,NaN,4.7,F2MR+R2,"{""الأحد"": ""١٠:٣٠ص–٢:٠٠م٧:٠٠–٩:٣٠م""}",سيغلق قريبًا · عند الساعة ٢ م · يفتح مجددًا عن...,NaN,سيهات,المنطقة الشرقية,26.484570,50.040013,https://www.google.com/maps?ftid=0x3e49ff826c0...,https://www.google.com/maps/place/%D9%85%D9%83...,14,خدمات مهنية وشخصية
2,2,مكتب زكي الخليفة للاستقدام Zaki Alkhalifa Recr...,جهة توظيف,شارع عمر بن عبدالعزيز، الديرة، سيهات‎ 31972,050 830 0554,NaN,3.9,F2PQ+W9,"{""الأحد"": ""٩:٠٠ص–١٢:٠٠م٤:٠٠–٩:٠٠م""}",مغلق · يفتح عند الساعة ٤ م,NaN,سيهات,المنطقة الشرقية,26.487369,50.038414,https://www.google.com/maps?ftid=0x3e49fe35e8a...,https://www.google.com/maps/place/%D9%85%D9%83...,14,خدمات مهنية وشخصية
3,3,شركة التهذيب للخدمات التعليمية,مؤسسة تعليمية,"Khaleej Rd, غرناطة، سيهات‎ 32434",013 333 6289,https://al-tahtheeb.com/,4.7,F3C4+PW,"{""الأحد"": ""٨:٠٠ص–٤:٠٠م""}",مفتوح · يغلق عند الساعة ٤ م,NaN,سيهات,المنطقة الشرقية,26.471858,50.057313,https://www.google.com/maps?ftid=0x3e49ffbbda8...,https://www.google.com/maps/place/%D8%B4%D8%B1...,15,تعليم
4,5,Al Mutawa Group,المقاول,2698 - Al Khisab Dist SAYHAT - 32452 -7037، سي...,013 856 1169,NaN,4.3,F2QP+G5,"{""الأحد"": ""٧:٠٠ص–٤:٠٠م""}",مفتوح · يغلق عند الساعة ٤ م,NaN,سيهات,المنطقة الشرقية,26.488822,50.035491,https://www.google.com/maps?ftid=0x3e49fe34f3d...,https://www.google.com/maps/place/Al+Mutawa+Gr...,14,إنشاءات وتحسين المنزل


# Visualization Using an Interactive Map
Create an interactive map showing all the scraped locations using folium.


In [48]:
# -*- coding: utf-8 -*-
"""
Interactive folium map of scraped Google Maps locations.

Features:
  - points coloured by `category_parent`
  - hover tooltip (name + fine category)
  - click popup with details and a clickable Google Maps link
  - one toggleable layer per category, shown as checkboxes

Run:  python map_locations.py     ->  writes locations_map.html
"""

LAT, LON = 'latitude', 'longitude'
PARENT_COL = 'category_parent'

# 20 reasonably distinct colours that read well on a light basemap.
PALETTE = [
    '#e6194B', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
    '#42d4f4', '#f032e6', '#469990', '#9A6324', '#800000',
    '#808000', '#000075', '#e67e22', '#1abc9c', '#2c3e50',
    '#c0392b', '#8e44ad', '#16a085', '#d35400', '#7f8c8d',
]
FALLBACK_COLOR = '#9e9e9e'        # grey, for unclassified points
UNCLASSIFIED = 'غير مصنّف'


def _has(value):
    """True if a value is present (not None / NaN / blank)."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return False
    return str(value).strip().lower() not in ('', 'nan', 'none')


def _row(label, value):
    """One HTML line for the popup, or '' if the value is missing."""
    if not _has(value):
        return ''
    return f'<div><b>{label}:</b> {html.escape(str(value).strip())}</div>'


def _link(label, url, text):
    """A clickable link row, or '' if the URL is missing."""
    if not _has(url):
        return ''
    href = str(url).strip()
    if not href.startswith(('http://', 'https://')):
        href = 'https://' + href
    return (f'<div><a href="{html.escape(href)}" target="_blank" '
            f'rel="noopener">{label}</a></div>')


def _popup_html(rec):
    name = html.escape(str(rec.get('name_ar', '')).strip()) or 'بدون اسم'
    parts = [
        f'<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif;'
        f'font-size:13px;line-height:1.6;min-width:220px">',
        f'<div style="font-size:15px;font-weight:700;margin-bottom:4px">{name}</div>',
        _row('التصنيف', rec.get('category')),
        _row('العنوان', rec.get('full_address')),
        _row('الهاتف', rec.get('phone_number')),
        _row('التقييم', rec.get('rating')),
        _row('نطاق السعر', rec.get('price_range')),
        _row('الحالة', rec.get('opening_status')),
        '<div style="margin-top:6px">',
        _link('🗺️ عرض على خرائط جوجل', rec.get('google_maps_url'), '🗺️ عرض على خرائط جوجل'),
        _link('🌐 الموقع الإلكتروني', rec.get('website'), '🌐 الموقع الإلكتروني'),
        '</div></div>',
    ]
    return ''.join(parts)


def _tooltip_html(rec):
    name = html.escape(str(rec.get('name_ar', '')).strip()) or 'بدون اسم'
    cat = html.escape(str(rec.get('category', '')).strip()) if _has(rec.get('category')) else ''
    sub = f'<div style="font-size:11px;color:#555">{cat}</div>' if cat else ''
    return f'<div dir="rtl" style="font-family:Tahoma,Arial,sans-serif"><b>{name}</b>{sub}</div>'


def build_map(df, lat=LAT, lon=LON, parent_col=PARENT_COL):
    """Build and return the folium map."""
    df = df.copy()
    df[lat] = pd.to_numeric(df[lat], errors='coerce')
    df[lon] = pd.to_numeric(df[lon], errors='coerce')
    df = df.dropna(subset=[lat, lon])

    groups = df[parent_col].fillna(UNCLASSIFIED)

    # colour per category: most common categories get the most distinct colours
    order = groups.value_counts().index.tolist()
    colors = {g: (PALETTE[i] if i < len(PALETTE) else FALLBACK_COLOR)
              for i, g in enumerate(order)}
    colors[UNCLASSIFIED] = FALLBACK_COLOR

    fmap = folium.Map(
        location=[df[lat].median(), df[lon].median()],
        zoom_start=11,
        tiles='cartodbpositron',          # light, low-contrast basemap
        control_scale=True,
    )

    # one FeatureGroup per category -> becomes one checkbox in the control
    layers = {g: folium.FeatureGroup(name=f'{g} ({(groups == g).sum()})')
              for g in order}

    for _, row in df.iterrows():
        rec = row.to_dict()
        g = rec.get(parent_col)
        g = g if _has(g) else UNCLASSIFIED
        color = colors.get(g, FALLBACK_COLOR)

        folium.CircleMarker(
            location=[rec[lat], rec[lon]],
            radius = (2 + rec.get('rating')), color=color, weight=1,
            fill=True, fill_color=color, fill_opacity=0.85,
            tooltip=folium.Tooltip(_tooltip_html(rec), sticky=True),
            popup=folium.Popup(_popup_html(rec), max_width=320),
        ).add_to(layers[g])

    for g in order:
        layers[g].add_to(fmap)

    folium.LayerControl(collapsed=False).add_to(fmap)   # checkbox panel
    fmap.fit_bounds([[df[lat].min(), df[lon].min()],
                     [df[lat].max(), df[lon].max()]])
    return fmap


In [49]:
fmap = build_map(df_with_parent)
fmap.save('locations_map.html')
print('Saved locations_map.html')

Saved locations_map.html


# Spatial Correlation Analysis

  

Helper functions:

In [50]:
LAT, LON = 'latitude', 'longitude'
RATING_COL = 'rating'
PARENT_COL = 'category_parent'
 
 
def project_to_metres(lat, lon):
    """Local equal-distance projection (good for a single city)."""
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    lat0, lon0 = lat.mean(), lon.mean()
    m_per_deg_lat = 111_320.0
    m_per_deg_lon = 111_320.0 * np.cos(np.radians(lat0))
    x = (lon - lon0) * m_per_deg_lon
    y = (lat - lat0) * m_per_deg_lat
    return np.column_stack([x, y])
 
 
def min_no_island_threshold(coords):
    """Smallest distance band so every point has >=1 neighbour (metres)."""
    tree = cKDTree(coords)
    nn_dist, _ = tree.query(coords, k=2)      # [self, nearest]
    return float(nn_dist[:, 1].max())
 
 
def _clean_coords(df):
    df = df.copy()
    df[LAT] = pd.to_numeric(df[LAT], errors='coerce')
    df[LON] = pd.to_numeric(df[LON], errors='coerce')
    return df.dropna(subset=[LAT, LON]).reset_index(drop=True)

# Calculating spatial correlation of ratings
### i.e., Do places with high ratings tend to be close together?

In [51]:
def morans_i_rating(df, threshold_m=1000, permutations=999):
    """Global Moran's I on rating (N/A ratings excluded)."""
    d = _clean_coords(df)
    d[RATING_COL] = pd.to_numeric(d[RATING_COL], errors='coerce')
    d = d.dropna(subset=[RATING_COL]).reset_index(drop=True)
 
    coords = project_to_metres(d[LAT], d[LON])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w = DistanceBand.from_array(coords, threshold=threshold_m, silence_warnings=True)
    w.transform = 'r'                          # row-standardised for Moran's I
 
    mi = Moran(d[RATING_COL].values, w, permutations=permutations)
    return {
        'n': int(w.n),
        'islands': len(w.islands),
        'morans_I': round(mi.I, 4),
        'expected_I': round(mi.EI, 4),
        'z_sim': round(mi.z_sim, 3),
        'p_sim': round(mi.p_sim, 4),
    }

In [52]:
THRESHOLD = 500   # metres
 
print("=== Moran's I on rating ===")
for k, v in morans_i_rating(df_with_parent, threshold_m=THRESHOLD).items():
    print(f"  {k}: {v}")



=== Moran's I on rating ===
  n: 1061
  islands: 8
  morans_I: 0.0001
  expected_I: -0.0009
  z_sim: 0.12
  p_sim: 0.431


morans_I = -0.0001  
+1 is perfect clustering, 0 is random, and −1 is a perfect checkerboard.  
The result is close to 0, which indicates there is no correlation; ratings are spread evenly.  


z_sim is the standard deviation of the observation from the expected (if points were spread randomly).  
p_sim is the p-score. i.e., the probability of getting a more extreme clustering than observed (if the points were shuffled randomly)


Since p_sim is 0.437, every time you shuffle the data randomly, there is a %43.7 percent chance of getting data that is more clustered.  

________________________________________________________________________________________________________

# Spatial Correlation of Catogories
### i.e., Do locations of the same category tend to be clustered together? If so, which categories?

In [53]:
def category_self_clustering(df, min_samples=60, threshold_m=1000, permutations=999):
    """Join-count self-clustering for each parent category with >= min_samples.
 
    Returns a DataFrame sorted so the most strongly clustered category is first.
    `bb_ratio` = observed like-like joins / expected under random labelling;
    >1 with a small p-value means that category clumps together.
    """
    d = _clean_coords(df)
    coords = project_to_metres(d[LAT], d[LON])
 
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        w = DistanceBand.from_array(coords, threshold=threshold_m, silence_warnings=True)
    w.transform = 'b'                          # binary weights for join counts
 
    counts = d[PARENT_COL].value_counts()
    cats = counts[counts >= min_samples].index.tolist()
 
    rows = []
    for cat in cats:
        y = (d[PARENT_COL] == cat).astype(int).values
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            jc = Join_Counts(y, w, permutations=permutations)
        expected = jc.mean_bb if jc.mean_bb else np.nan
        rows.append({
            'category': cat,
            'n': int(y.sum()),
            'observed_BB': round(float(jc.bb), 1),
            'expected_BB': round(float(jc.mean_bb), 1),
            'bb_ratio': round(float(jc.bb / expected), 2) if expected else np.nan,
            'p_sim': round(float(jc.p_sim_bb), 4),
        })
 
    res = pd.DataFrame(rows)
    res['clusters'] = (res['bb_ratio'] > 1) & (res['p_sim'] <= 0.05)
    return res.sort_values('bb_ratio', ascending=False).reset_index(drop=True)

In [55]:
res = category_self_clustering(df_with_parent, min_samples=60, threshold_m=1000)
res

,category,n,observed_BB,expected_BB,bb_ratio,p_sim,clusters
0,تجزئة وتسوق,163,3613.0,2472.8,1.46,0.001,True
1,صحة وطب,60,446.0,333.6,1.34,0.023,True
2,مطاعم ومقاهي,138,2368.0,1776.7,1.33,0.001,True
3,تجميل وعناية شخصية,137,2172.0,1754.4,1.24,0.007,True
4,خدمات مهنية وشخصية,104,1199.0,1004.4,1.19,0.040,True
5,أماكن دينية,67,456.0,416.8,1.09,0.238,False
6,تعليم,82,606.0,622.1,0.97,0.578,False
7,بقالة وأسواق غذائية,67,379.0,414.2,0.91,0.743,False
8,سيارات ومركبات,106,765.0,1046.9,0.73,1.000,False
9,رياضة وترفيه,83,400.0,639.8,0.63,1.000,False
